# 4장 1강 파이토치 기초 및 심층 신경망

In [7]:
# SGD 옵티마이저 활용한 다층 퍼셉트론 구현
# SGD (확률적 경사하강법) : 역전파가 준 편미분 값에 내가 가진 학습률만 곱해서 바로 출력
# 역전파 : 어느 방향으로 얼만큼 가파른지(편미분) 정보만 컴퓨터에 알려주는 설계도
# 옵티마이저 : 실제 몇 걸음 걸어갈지 (learning rate; 학습률) 결정하고 발을 내딛는 행동 대장

# 필요한 도구 불러오기
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader, random_split



In [8]:
# 데이터 셋 다운로드 및 분할
raw_train_data = datasets.FashionMNIST(
    root="data", train=True, download=True, transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data", train=False, download=True, transform=ToTensor()
)

train_size = 50000
val_size = 10000

train_data, val_data = random_split(
    raw_train_data, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"데이터 분할 완료 -> 훈련용: {len(train_data)}장 | 검증용: {len(val_data)}장 | 테스트용: {len(test_data)}장 ")

train_ldr = DataLoader(train_data, batch_size=32, shuffle=True)
val_ldr = DataLoader(val_data, batch_size=32, shuffle=False)
test_ldr = DataLoader(test_data, batch_size=32, shuffle=False)

데이터 분할 완료 -> 훈련용: 50000장 | 검증용: 10000장 | 테스트용: 10000장 


In [9]:
# 모델 설계 (인공싱경망 구조 정의)

import torch.nn as nn
import torch.optim as optim

class AdvancedFashionClassifier(nn.Module):
    def __init__(self):
        super(AdvancedFashionClassifier, self).__init__()
        # 2D 이미지 28*28 -> 1D 벡터 784 형태로 펼쳐주는 계층
        self.flatten = nn.Flatten()
        # 입력층 (784) -> 은닉층 (64) 선형 변환
        self.linear1 = nn.Linear(28*28, 64)
        # 활성화 함수로 은닉층에 비선형성 주입 - ReLU 지정
        self.relu = nn.ReLU()
        # 학습 중 뉴런의 20%를 무작위로 비활성화 - dropout 설정
        self.dropout = nn.Dropout(p=0.2)
        # 은닉층(64) -> 출력층(10)
        self.linear2 = nn.Linear(64,10)

    def forward(self, x):
        out = self.flatten(x)
        out = self.linear1(out)
        out = self.relu(out)
        out = self.dropout(out) # 훈련시에만 적용
        out = self.linear2(out) # 별도의 softmax없이 raw 점수 (logits)
        return out

model = AdvancedFashionClassifier().to(device='cpu')

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"모델의 가속기 이관 완료 | 학습 가능한 총 파라미터 수: {total_params:,}개")

모델의 가속기 이관 완료 | 학습 가능한 총 파라미터 수: 50,890개


In [10]:
# 손실 함수와 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [11]:
# 미니 배치 크기 변경 및 5에포크 학습 루프 구현
# 배치 사이즈 32 -> 64 / 셔플 true

train_ldr = DataLoader(train_data, batch_size=64, shuffle=True)
val_ldr = DataLoader(val_data, batch_size=64, shuffle=False)
test_ldr = DataLoader(test_data, batch_size=64, shuffle=False)

In [12]:
# 하이퍼 파라미터 및 변수 설정
epochs = 5
patience = 3
patience_counter = 0
best_val_loss = float('inf')

train_losses = []
val_losses = []

print("\n=====모델 학습 및 검증을 시작합니다.=====")
for epoch in range(epochs):
    # 훈련 모드
    running_loss = 0.0
    model.train()

    for images, labels in train_ldr:
        # 미니배치 데이터 -> 모델이 위치한 동일 가속기 메모리로 이관
        images = images.to(device='cpu')
        labels = labels.to(device='cpu')

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_ldr)
    train_losses.append(epoch_loss)

    # 검증 모드
    model.eval()
    val_loss = 0.0

    with torch.no_grad(): # 검증 시 미분 계산 제한하여 연산 자원 최적화
        for images, labels in val_ldr:
            images = images.to(device='cpu')
            labels = labels.to(device='cpu')

            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    epoch_val_loss = val_loss / len(val_ldr)
    val_losses.append(epoch_val_loss)

    print(f"Epoch[{epoch+1}/{epochs}] | 훈련 손실: {epoch_loss:.4f} | 검증 손실: {epoch_val_loss:.4f}")

    # 조기 종료 조건 검증 로직
    if epoch_val_loss < best_val_loss:
        # 검증 손실이 기존 최적값보다 개선된 경우
        best_val_loss = epoch_val_loss
        patience_counter = 0
        # 최적 모델 가중치 파일로 저장
        torch.save(model.state_dict(), "best_fashion_model.pth")
        print(f"=> 최적 모델 저장 완료 (최저 검증 손실): {best_val_loss:.4f}")

    else:
        # 검증 손실이 개선되지 않는 경우
        patience_counter += 1
        print(f"=> 검증 손실 미개선 (조기 종료 대기 카운트: {patience_counter}/{patience})")

        # 설정한 참을성 한계를 초과하면 학습을 강제 중던
        if patience_counter >= patience:
            print(f"\n[Early Stopping 활성화] {epoch+1} 에포크에서 학습을 조기 종료합니다.")
            break




=====모델 학습 및 검증을 시작합니다.=====
Epoch[1/5] | 훈련 손실: 1.2600 | 검증 손실: 0.8292
=> 최적 모델 저장 완료 (최저 검증 손실): 0.8292
Epoch[2/5] | 훈련 손실: 0.7885 | 검증 손실: 0.6830
=> 최적 모델 저장 완료 (최저 검증 손실): 0.6830
Epoch[3/5] | 훈련 손실: 0.6803 | 검증 손실: 0.6162
=> 최적 모델 저장 완료 (최저 검증 손실): 0.6162
Epoch[4/5] | 훈련 손실: 0.6194 | 검증 손실: 0.5656
=> 최적 모델 저장 완료 (최저 검증 손실): 0.5656
Epoch[5/5] | 훈련 손실: 0.5817 | 검증 손실: 0.5415
=> 최적 모델 저장 완료 (최저 검증 손실): 0.5415


In [14]:
# (심화) 은닉 뉴런 256개 및 드롭아웃 확률 50퍼센트 설정

# 새로운 모델 설계 (인공싱경망 구조 정의)

import torch.nn as nn
import torch.optim as optim

class AdvancedFashionClassifier_2(nn.Module):
    def __init__(self):
        super(AdvancedFashionClassifier_2, self).__init__()
        # 2D 이미지 28*28 -> 1D 벡터 784 형태로 펼쳐주는 계층
        self.flatten = nn.Flatten()
        # 입력층 (784) -> 은닉층 (256) 선형 변환
        self.linear1 = nn.Linear(28*28, 256)
        # 활성화 함수로 은닉층에 비선형성 주입 - ReLU 지정
        self.relu = nn.ReLU()
        # 학습 중 뉴런의 50%를 무작위로 비활성화 - dropout 설정
        self.dropout = nn.Dropout(p=0.5)
        # 은닉층(256) -> 출력층(10)
        self.linear2 = nn.Linear(256,10)

    def forward(self, x):
        out = self.flatten(x)
        out = self.linear1(out)
        out = self.relu(out)
        out = self.dropout(out) # 훈련시에만 적용
        out = self.linear2(out) # 별도의 softmax없이 raw 점수 (logits)
        return out

model = AdvancedFashionClassifier_2().to(device='cpu')

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[클래스v2] 모델의 가속기 이관 완료 | 학습 가능한 총 파라미터 수: {total_params:,}개")

[클래스v2] 모델의 가속기 이관 완료 | 학습 가능한 총 파라미터 수: 203,530개
